# Binning Continuous Variables for Binary Prediction

This notebook surveys the main approaches to **discretising (binning) a continuous predictor** when the outcome is a binary target variable (e.g., default / no-default, churn / no-churn). It covers three broad traditions:

1. **Classical / statistical approaches** – equal-width, equal-frequency, natural breaks
2. **Machine-learning approaches** – decision-tree splitting, MDLP, k-means
3. **Credit-scorecard approaches** – Weight of Evidence (WoE) / Information Value (IV) monotonic binning, ChiMerge

For each approach the notebook presents:
- the core idea and algorithm
- the key formula(s)
- merits and limitations
- a comparison table at the end

> **Note**: This notebook is explanatory / reference material. Code cells contain illustrative pseudocode or short Python sketches rather than fully executed outputs.

---
## 0. Setup and notation

Throughout the notebook:

| Symbol | Meaning |
|--------|---------|
| $x$ | continuous predictor variable |
| $y \in \{0,1\}$ | binary target (1 = event, 0 = non-event) |
| $n$ | total number of observations |
| $n_1$ | total number of events ($y=1$) |
| $n_0$ | total number of non-events ($y=0$) |
| $B$ | number of bins |
| $b$ | bin index, $b = 1,\dots,B$ |
| $n_b$ | number of observations in bin $b$ |
| $d_b$ | number of events in bin $b$ |
| $p_b = d_b/n_b$ | event rate ("bad rate") in bin $b$ |
| $\bar p = n_1/n$ | overall event rate |

---
## 1. Statistical Approaches

Statistical approaches are the oldest and simplest. They rely on the **distribution of $x$ alone** (unsupervised), with no reference to $y$. They are easy to explain to non-technical stakeholders and require minimal computation.

### 1.1 Equal-Width (Uniform) Binning

**Idea**: Divide the range $[x_{\min}, x_{\max}]$ into $B$ intervals of equal width.

$$
w = \frac{x_{\max} - x_{\min}}{B}
$$

Cut-points: $c_b = x_{\min} + b \cdot w,\quad b = 0, 1, \dots, B$.

**Algorithm**:
1. Choose $B$ (or use Sturges' rule $B = \lceil \log_2 n \rceil + 1$, or Scott's rule, or Freedman–Diaconis).
2. Compute $w$.
3. Assign each observation to $\lfloor (x_i - x_{\min}) / w \rfloor + 1$.

**Merits**:
- Trivially simple; $O(n)$ time.
- Intuitive to communicate ("age 20–30, 30–40, …").
- Preserves the numeric scale.

**Limitations**:
- Highly sensitive to outliers (a single extreme value stretches all bins).
- Produces very unequal bin counts for skewed distributions.
- Ignores the target variable entirely — bins may not separate $y=1$ from $y=0$.

### 1.2 Equal-Frequency (Quantile) Binning

**Idea**: Each bin contains approximately the same number of observations, $n/B$.

Cut-points are the $(k/B)$-th quantiles of $x$:
$$
c_b = Q\!\left(\frac{b}{B}\right), \quad b = 0, 1, \dots, B
$$

where $Q(\cdot)$ is the empirical quantile function.

**Merits**:
- Robust to outliers and skew.
- Every bin has enough data to estimate event rates reliably.
- Widely used as a first-pass step before supervised refinement.

**Limitations**:
- Still ignores $y$ — bins may mix events and non-events.
- Identical values at cut-points cause ambiguous bin assignment (ties).
- The number of bins $B$ must be chosen a priori.

### 1.3 Natural Breaks (Jenks Optimisation)

**Idea**: Choose cut-points that minimise within-bin variance while maximising between-bin variance — a 1-D $k$-means applied to the sorted values of $x$.

**Objective** (minimise):
$$
\text{SDCM} = \sum_{b=1}^{B} \sum_{i \in \text{bin}_b} (x_i - \bar x_b)^2
$$

The Goodness-of-Variance Fit (GVF) measures how good the solution is:
$$
\text{GVF} = 1 - \frac{\text{SDCM}}{\text{SDAM}}
$$
where $\text{SDAM} = \sum_i (x_i - \bar x)^2$ is the total variance.

Solved exactly in $O(Bn^2)$ time via dynamic programming (Fisher 1958), or approximately via iterative heuristics.

**Merits**:
- Finds "natural" clusters in the data.
- Popular in cartography and spatial statistics.
- GVF provides an objective measure of bin quality.

**Limitations**:
- More computationally expensive than equal-width / equal-frequency.
- Still unsupervised — natural clusters in $x$ may not align with $y$.
- Less common in credit/insurance risk contexts.

---
## 2. Machine-Learning Approaches

ML approaches use the binary label $y$ to drive the bin boundaries. They are **supervised** and typically produce bins that are more predictive of the outcome.

### 2.1 Decision-Tree (Top-Down) Splitting

**Idea**: Train a single-feature decision tree (stump or deeper) on $x$ with $y$ as the target. Each leaf defines a bin.

The tree chooses the cut-point $c^*$ that maximises an impurity reduction. For **Gini impurity**:

$$
\text{Gini}(t) = 1 - \sum_{k} p_k^2
$$

For a binary target, $\text{Gini}(t) = 2 p_t(1-p_t)$ where $p_t$ is the event rate in node $t$.

The **Gini gain** of split $c$:
$$
\Delta G(c) = \text{Gini}(\text{parent}) - \frac{n_L}{n}\,\text{Gini}(L) - \frac{n_R}{n}\,\text{Gini}(R)
$$

Equivalently, splits can be chosen by **entropy** (information gain) or **$\chi^2$ statistic**.

**For entropy / information gain**:
$$
H(t) = -\sum_k p_k \log_2 p_k, \qquad \Delta H(c) = H(\text{parent}) - \frac{n_L}{n}H(L) - \frac{n_R}{n}H(R)
$$

**Algorithm**:
1. Sort observations by $x$.
2. Evaluate all candidate split points.
3. Choose $c^*$ with maximum gain; recurse on each side until a stopping criterion (max depth, min samples per leaf, etc.).
4. Extract leaf boundaries as bin cut-points.

**Merits**:
- Fully supervised — bins directly optimise discriminatory power.
- Handles non-monotone relationships naturally.
- Built into sklearn (`DecisionTreeClassifier` with `max_features=1`).

**Limitations**:
- Tends to overfit without depth/sample constraints.
- Does not guarantee monotone event rates across bins.
- Bin boundaries can change substantially with small data perturbations.

### 2.2 MDLP — Minimum Description Length Principle

**Idea** (Fayyad & Irani 1993): A top-down recursive split is accepted only if it reduces the **total description length** of the dataset. The stopping criterion replaces arbitrary depth/sample parameters with an information-theoretic criterion.

A split at $c$ is accepted if:
$$
\text{Gain}(c) > \frac{\log_2(n-1)}{n} + \frac{\Delta(c)}{n}
$$

where
$$
\Delta(c) = \log_2(3^k - 2) - \bigl[k\,H(S) - k_L\,H(S_L) - k_R\,H(S_R)\bigr]
$$

and $k, k_L, k_R$ are the number of distinct classes in the parent and two child nodes respectively ($k=2$ for binary targets).

**Merits**:
- Parameter-free (no need to choose $B$).
- Theoretically grounded — prevents over-discretisation.
- Available via `mlxtend` (`mdlp_discretization`).

**Limitations**:
- Can produce very few bins (sometimes just 1) when the signal is weak.
- Designed for nominal classes; binary case is straightforward but multi-class adaptation is less obvious.
- Less interpretable than WoE in a scorecard context.

### 2.3 ChiMerge (Bottom-Up Merging)

**Idea** (Kerber 1992): Start with one bin per unique value of $x$, then iteratively **merge** the pair of adjacent bins whose $\chi^2$ statistic is smallest (i.e., the pair that is most similar in terms of class distribution). Stop when all adjacent pairs have a $\chi^2$ above a threshold.

For two adjacent bins $b$ and $b+1$, the $\chi^2$ statistic is:
$$
\chi^2 = \sum_{j \in \{0,1\}} \sum_{b' \in \{b, b+1\}} \frac{(O_{b'j} - E_{b'j})^2}{E_{b'j}}
$$

where $O_{b'j}$ is the observed count and $E_{b'j} = n_{b'} \cdot (n_j / (n_b + n_{b+1}))$ is the expected count under independence.

**Algorithm**:
1. Sort unique values of $x$; create one bin per value.
2. Compute $\chi^2$ for each pair of adjacent bins.
3. Merge the pair with the smallest $\chi^2$.
4. Repeat until the minimum $\chi^2 \geq \chi^2_{\alpha, df=1}$ (e.g., $\alpha=0.05 \Rightarrow 3.84$).

**Merits**:
- Bottom-up, so bins represent genuinely similar segments.
- The stopping criterion is statistically principled.
- Naturally handles sparse regions.

**Limitations**:
- $O(n^2)$ worst-case — slow on large datasets without binning initial quantiles first.
- Does not enforce monotonicity.
- The $\chi^2$ threshold is another hyperparameter (though interpretable as a significance level).

---
## 3. Credit-Scorecard Approaches

In retail credit risk modelling (Basel IRB, scorecards), **Weight of Evidence (WoE)** binning is the dominant paradigm. The goals are:

1. Bins with **monotone event rates** (so that higher/lower score = lower/higher risk).
2. Sufficient observations per bin for stable WoE estimates.
3. A summary statistic (**Information Value, IV**) to rank predictors.
4. Interpretability for regulatory review.

### 3.1 Weight of Evidence (WoE)

For bin $b$:

$$
\text{WoE}_b = \ln\!\left(\frac{d_b / n_1}{(n_b - d_b) / n_0}\right) = \ln\!\left(\frac{\text{Distribution of events}_b}{\text{Distribution of non-events}_b}\right)
$$

Equivalently:
$$
\text{WoE}_b = \ln\!\left(\frac{p_b}{1 - p_b}\right) - \ln\!\left(\frac{\bar p}{1 - \bar p}\right) = \text{logit}(p_b) - \text{logit}(\bar p)
$$

This reveals that WoE is the **log-odds of the bin minus the overall log-odds**. When $x$ is replaced by its WoE values, a logistic regression coefficient is forced to 1 — it purely captures the rank-order of bins.

**Interpretation**:
| WoE | Meaning |
|-----|---------|
| $> 0$ | Bin has higher event rate than average |
| $= 0$ | Bin has exactly average event rate |
| $< 0$ | Bin has lower event rate than average |

### 3.2 Information Value (IV)

IV summarises the total predictive power of variable $x$ across all bins:

$$
\text{IV} = \sum_{b=1}^{B} \left(\frac{d_b}{n_1} - \frac{n_b - d_b}{n_0}\right) \times \text{WoE}_b
$$

This is the **Kullback–Leibler divergence** (summed symmetrically) between the event and non-event distributions:
$$
\text{IV} = \sum_b \left(\text{Distr. Events}_b - \text{Distr. Non-Events}_b\right) \ln\!\frac{\text{Distr. Events}_b}{\text{Distr. Non-Events}_b}
$$

**Rule of thumb for IV** (Siddiqi 2006):

| IV range | Predictive power |
|----------|------------------|
| $< 0.02$ | Useless |
| $0.02 – 0.10$ | Weak |
| $0.10 – 0.30$ | Medium |
| $0.30 – 0.50$ | Strong |
| $> 0.50$ | Suspicious (check for leakage) |

### 3.3 Monotonic Binning Algorithms

The standard credit-scorecard requirement is **monotone WoE** (or equivalently monotone event rates) across bins ordered by the predictor value. Several algorithms achieve this:

#### 3.3.1 Quantile + Isotonic Regression (Pool Adjacent Violators)

1. Start with equal-frequency bins (e.g., 20 quantile bins).
2. Compute the event rate $p_b$ for each bin.
3. Apply the **Pool Adjacent Violators Algorithm (PAVA)** to merge bins that violate monotonicity:

   - Scan bins left to right.
   - When $p_b > p_{b+1}$ (for desired increasing order), merge bins $b$ and $b+1$ into one with pooled rate $\hat p = (d_b + d_{b+1})/(n_b + n_{b+1})$.
   - Repeat until all adjacent rates are monotone.

The resulting sequence satisfies the **isotonic regression** criterion:
$$
\hat{\mathbf{p}} = \arg\min_{\mathbf{q}: q_1 \leq \cdots \leq q_B} \sum_b n_b (p_b - q_b)^2
$$

This is equivalent to a least-squares projection onto the monotone cone.

#### 3.3.2 Optimal Monotone Binning via Dynamic Programming

For a given number of bins $B^*$, find the cut-points that maximise IV subject to monotonicity. This can be solved in $O(B n)$ time using DP on pre-computed quantile bin statistics.

#### 3.3.3 Packages

| Package | Key functions |
|---------|---------------|
| `scorecardpy` | `sc.woebin()` — auto monotonic WoE binning |
| `optbinning` | `OptimalBinning` — MILP-based optimal binning |
| `xverse` | `WOE` transformer |
| `sklearn` | `KBinsDiscretizer` (quantile strategy) + manual PAVA |

### 3.4 Scorecard-Specific Considerations

Beyond the algorithm, practitioners in credit risk apply additional constraints:

1. **Minimum bin size**: Typically ≥ 5% of total observations per bin to ensure stable WoE estimates.
2. **Separate missing-value bin**: Missing values are placed in their own bin with its own WoE, preserving information about missingness patterns.
3. **Business interpretability**: Bins should align with natural business categories (e.g., age < 25, 25–35, …) even if this slightly reduces IV.
4. **Stability over time**: Population Stability Index (PSI) is used to detect bin drift:
$$
\text{PSI} = \sum_b \left(p_b^{\text{train}} - p_b^{\text{test}}\right) \ln\!\frac{p_b^{\text{train}}}{p_b^{\text{test}}}
$$
5. **Regulatory review**: Cut-points must be defensible to auditors and regulators; complex ML boundaries are harder to justify than simple quantile or WoE bins.

---
## 4. Comparison of Approaches

### 4.1 Algorithm Summary

| Method | Supervised | Monotone | Objective | Complexity |
|--------|-----------|---------|-----------|------------|
| Equal-width | No | No | Uniform range | $O(n)$ |
| Equal-frequency | No | No | Uniform count | $O(n \log n)$ |
| Natural breaks (Jenks) | No | No | Min within-bin variance | $O(Bn^2)$ |
| Decision tree | Yes | No | Gini / entropy gain | $O(n \log n)$ |
| MDLP | Yes | No | Description length | $O(n \log n)$ |
| ChiMerge | Yes | No | $\chi^2$ similarity | $O(n^2)$ |
| WoE monotonic (PAVA) | Yes | Yes | Isotonic regression | $O(n)$ after quantiles |
| Optimal WoE (DP/MILP) | Yes | Yes | Maximise IV | $O(Bn)$ |

### 4.2 Key Formula Comparison

| Method | Core formula |
|--------|--------------|
| Equal-width cut | $c_b = x_{\min} + b \cdot (x_{\max}-x_{\min})/B$ |
| Quantile cut | $c_b = Q(b/B)$ |
| Jenks objective | $\text{SDCM} = \sum_b\sum_{i \in b}(x_i - \bar x_b)^2$ |
| Tree Gini gain | $\Delta G = 2p(1-p) - \tfrac{n_L}{n}2p_L(1-p_L) - \tfrac{n_R}{n}2p_R(1-p_R)$ |
| MDLP threshold | $\text{Gain} > [\log_2(n-1) + \Delta]/n$ |
| ChiMerge $\chi^2$ | $\chi^2 = \sum_{j,b}(O_{bj}-E_{bj})^2/E_{bj}$ |
| WoE | $\ln(d_b/n_1) - \ln((n_b-d_b)/n_0)$ |
| IV | $\sum_b(d_b/n_1 - (n_b-d_b)/n_0) \cdot \text{WoE}_b$ |
| PAVA objective | $\min_{\mathbf{q} \text{ isotone}} \sum_b n_b(p_b-q_b)^2$ |
| PSI | $\sum_b(p_b^{\rm train}-p_b^{\rm test})\ln(p_b^{\rm train}/p_b^{\rm test})$ |

### 4.3 Merits and Limitations

| Dimension | Statistical | Decision Tree / MDLP | WoE Monotonic |
|-----------|------------|----------------------|---------------|
| **Predictive power** | Low (unsupervised) | Medium–High | High (when signal exists) |
| **Interpretability** | High | Medium | High (log-odds units) |
| **Monotonicity** | Not guaranteed | Not guaranteed | Enforced |
| **Missing values** | Ad hoc | Depends on implementation | Separate bin, WoE computed |
| **Regulation / audit** | Easy to defend | Moderate | Widely accepted in Basel IRB |
| **Overfitting risk** | Low | Medium (needs tuning) | Low (monotone constraint) |
| **Requires $B$** | Yes | No (MDLP) / Yes (tree depth) | No (PAVA merges until monotone) |
| **Scalability** | Excellent | Good | Good |
| **Non-linear patterns** | Captured by bins | Yes | Only if monotone |
| **Typical use case** | EDA, dashboards | General ML features | Credit scorecards, risk models |

---
## 5. Relationship Between WoE and Logistic Regression

A key insight from the credit-scorecard field is the direct link between WoE binning and logistic regression:

If we replace the predictor $x$ with its WoE-encoded value $w = \text{WoE}(x)$, then in the logistic regression
$$
\log\!\frac{P(y=1|w)}{P(y=0|w)} = \alpha + \beta w
$$
the MLE of $\beta$ equals 1 (up to scaling) when WoE is computed from the same data. In other words, **WoE transformation and logistic regression are equivalent on training data** — the logistic model after WoE transformation is a linear scorecard by construction.

More formally, if $p_b^{\text{model}}$ is the predicted probability in bin $b$:
$$
\log\!\frac{p_b^{\text{model}}}{1 - p_b^{\text{model}}} = \alpha + 1 \cdot \text{WoE}_b
$$
which satisfies $p_b^{\text{model}} = p_b$ exactly on training data. The scorecard value for bin $b$ is then proportional to $\text{WoE}_b$, scaled to a user-friendly integer point range (e.g., 0–1000).

---
## 6. Practical Guidance

### When to use each approach

```
Problem context
│
├─ Exploratory data analysis / dashboards
│   └─► Equal-width or equal-frequency bins
│
├─ General ML pipeline (gradient boosting, random forest)
│   └─► Decision-tree binning or MDLP (rare; tree models handle
│       continuous features natively)
│
├─ Logistic regression without scorecard requirement
│   └─► ChiMerge or decision-tree binning (for non-linear
│       relationship capture)
│
└─ Credit scorecard / regulatory model
    └─► WoE monotonic binning (PAVA or optimal DP)
        ├─ Use scorecardpy.woebin() or optbinning.OptimalBinning
        ├─ Check IV > 0.10 for variable selection
        ├─ Monitor PSI < 0.10 on holdout / production data
        └─ Document cut-points and business rationale
```

### Common pitfalls

1. **Leakage**: WoE computed on the full dataset (including test set) inflates IV — always fit bins on training data only.
2. **Too many bins**: Increases IV mechanically; use cross-validation or holdout to select the number of bins.
3. **Zero counts in a bin**: Leads to $\text{WoE} = \pm\infty$; apply Laplace smoothing or merge with an adjacent bin.
4. **Non-monotone WoE after new data arrives**: Retrain or rebin; monitor with PSI.
5. **Ignoring business constraints**: Purely optimal bins may conflict with regulatory or business definitions.

---
## 7. Pseudocode Sketches

These cells are illustrative — they are not meant to be executed. For working implementations, see the packages listed in Section 3.3.3.

In [ ]:
# ── Pseudocode: Equal-frequency binning ──────────────────────────────────────
import numpy as np
import pandas as pd

def equal_freq_bins(x, B=10):
    """Return B equal-frequency cut-points for array x."""
    quantiles = np.linspace(0, 100, B + 1)          # 0%, 10%, …, 100%
    cuts = np.percentile(x, quantiles)
    cuts = np.unique(cuts)                           # remove duplicate quantiles
    return cuts

# Usage:
# cuts = equal_freq_bins(df['age'], B=10)
# df['age_bin'] = pd.cut(df['age'], bins=cuts, include_lowest=True)

In [ ]:
# ── Pseudocode: WoE and IV calculation ──────────────────────────────────────
import numpy as np
import pandas as pd

def compute_woe_iv(df, feature_col, target_col, bin_col):
    """
    Compute WoE and IV for a pre-binned feature.

    Parameters
    ----------
    df : pd.DataFrame
    feature_col : str  (original continuous feature, for reference)
    target_col  : str  (binary 0/1)
    bin_col     : str  (pre-computed bin labels)
    """
    n1 = df[target_col].sum()          # total events
    n0 = len(df) - n1                  # total non-events

    stats = (
        df.groupby(bin_col, observed=False)[target_col]
        .agg(n='size', events='sum')
        .assign(non_events=lambda d: d['n'] - d['events'])
    )
    eps = 0.5  # Laplace smoothing to avoid log(0)
    stats['distr_events']     = (stats['events']     + eps) / (n1 + eps * len(stats))
    stats['distr_non_events'] = (stats['non_events'] + eps) / (n0 + eps * len(stats))

    stats['woe'] = np.log(stats['distr_events'] / stats['distr_non_events'])
    stats['iv']  = (stats['distr_events'] - stats['distr_non_events']) * stats['woe']

    total_iv = stats['iv'].sum()
    return stats, total_iv

# Usage:
# stats, iv = compute_woe_iv(train_df, 'age', 'default', 'age_bin')
# print(f'IV = {iv:.4f}')

In [ ]:
# ── Pseudocode: Pool Adjacent Violators (PAVA) for monotone binning ──────────
import numpy as np

def pava_merge(bin_events, bin_non_events, direction='increasing'):
    """
    Pool Adjacent Violators Algorithm.

    Parameters
    ----------
    bin_events     : list of int  — event counts per initial bin
    bin_non_events : list of int  — non-event counts per initial bin
    direction      : 'increasing' or 'decreasing' monotone event rate

    Returns
    -------
    merged_events, merged_non_events : lists after merging
    """
    e = list(bin_events)
    ne = list(bin_non_events)

    changed = True
    while changed:
        changed = False
        i = 0
        while i < len(e) - 1:
            rate_i   = e[i]   / (e[i]   + ne[i]   + 1e-9)
            rate_i1  = e[i+1] / (e[i+1] + ne[i+1] + 1e-9)
            violates = (rate_i > rate_i1) if direction == 'increasing' else (rate_i < rate_i1)
            if violates:
                # Merge bins i and i+1
                e[i]  = e[i]  + e[i+1]
                ne[i] = ne[i] + ne[i+1]
                del e[i+1]
                del ne[i+1]
                changed = True
            else:
                i += 1
    return e, ne

In [ ]:
# ── Pseudocode: Decision-tree binning ────────────────────────────────────────
# from sklearn.tree import DecisionTreeClassifier
# import numpy as np

def tree_bin_cutpoints(x, y, max_leaf_nodes=10, min_samples_leaf=50):
    """
    Derive bin cut-points from a single-feature decision tree.
    """
    # tree = DecisionTreeClassifier(
    #     max_leaf_nodes=max_leaf_nodes,
    #     min_samples_leaf=min_samples_leaf
    # )
    # tree.fit(x.reshape(-1, 1), y)
    # thresholds = tree.tree_.threshold
    # cuts = sorted(set(thresholds[thresholds != -2]))  # -2 marks leaf nodes
    # return cuts
    pass  # placeholder — uncomment sklearn lines above to run

# Usage:
# cuts = tree_bin_cutpoints(df['age'].values, df['default'].values)
# df['age_bin'] = pd.cut(df['age'], bins=[-np.inf] + cuts + [np.inf])

---
## 8. References

1. **Fayyad, U. M. & Irani, K. B.** (1993). Multi-interval discretization of continuous-valued attributes for classification learning. *IJCAI-93*.
2. **Kerber, R.** (1992). ChiMerge: Discretization of numeric attributes. *AAAI-92*.
3. **Fisher, W. D.** (1958). On grouping for maximum homogeneity. *Journal of the American Statistical Association*, 53(284), 789–798.
4. **Siddiqi, N.** (2006). *Credit Risk Scorecards: Developing and Implementing Intelligent Credit Scoring*. Wiley.
5. **Anderson, R.** (2007). *The Credit Scoring Toolkit*. Oxford University Press.
6. **Mironchyk, P. & Tchistiakov, V.** (2017). Monotone optimal binning algorithm for credit risk modeling. *SSRN*.
7. **Navas-Palencia, G.** (2020). Optimal binning: Mathematical programming formulation. *arXiv:2001.08025*.
8. **Dougherty, J., Kohavi, R. & Sahami, M.** (1995). Supervised and unsupervised discretization of continuous features. *ICML-95*.
9. **Garcia, S., Luengo, J. & Herrera, F.** (2013). A survey of discretization techniques. *IEEE TKDE*, 25(4).
10. **sklearn documentation**: `KBinsDiscretizer`, `DecisionTreeClassifier` — https://scikit-learn.org